## Test Camera - Color

IMX378 Sensor 2 on middle of camera

In [3]:
import depthai as dai
import cv2
import time

pipeline = dai.Pipeline()

cam = pipeline.create(dai.node.Camera).build(
    dai.CameraBoardSocket.CAM_A
)

rgb = cam.requestOutput(
    size=(1280, 720),
    type=dai.ImgFrame.Type.BGR888p,
    fps=30
)

q = rgb.createOutputQueue(
    maxSize=4,
    blocking=False
)

pipeline.start()

print("OAK-D RGB started")
print("Press Q to quit")

frame_count = 0
start_time = time.time()
fps = 0.0

while pipeline.isRunning():

    packet = q.get()
    frame = packet.getCvFrame()

    frame_count += 1
    elapsed = time.time() - start_time

    if elapsed >= 1.0:
        fps = frame_count / elapsed
        print(f"FPS: {fps:.2f}")

        frame_count = 0
        start_time = time.time()

    cv2.putText(
        frame,
        f"FPS: {fps:.1f}",
        (20, 40),
        cv2.FONT_HERSHEY_SIMPLEX,
        1,
        (255, 255, 255),
        2
    )

    cv2.imshow("OAK-D RGB Test", frame)

    if cv2.waitKey(1) & 0xFF == ord("q"):
        break

pipeline.stop()
cv2.destroyAllWindows()

OAK-D RGB started
Press Q to quit
FPS: 15.17
FPS: 15.64
FPS: 15.11
FPS: 15.68
FPS: 15.67
FPS: 15.50
FPS: 15.70
FPS: 15.73
FPS: 15.80
FPS: 15.83
FPS: 15.29
FPS: 1.57


: 

# Test Monochrome + RGB 
all camera in one window

In [ ]:
import depthai as dai
import cv2
import numpy as np
import time


# ============================================================
# SETTINGS
# ============================================================

WINDOW_NAME = "OAK-D - Left | RGB | Right"

WINDOW_WIDTH = 1280
WINDOW_HEIGHT = 720

CAMERA_WIDTH = 640
CAMERA_HEIGHT = 480
CAMERA_FPS = 30


# ============================================================
# PIPELINE
# ============================================================

pipeline = dai.Pipeline()


# ------------------------------------------------------------
# CENTER RGB CAMERA
# CAM_A = RGB camera
# ------------------------------------------------------------

rgbCam = pipeline.create(dai.node.Camera).build(
    dai.CameraBoardSocket.CAM_A
)

rgbOutput = rgbCam.requestOutput(
    size=(CAMERA_WIDTH, CAMERA_HEIGHT),
    type=dai.ImgFrame.Type.BGR888p,
    fps=CAMERA_FPS
)


# ------------------------------------------------------------
# LEFT MONO CAMERA
# ------------------------------------------------------------

leftCam = pipeline.create(dai.node.Camera).build(
    dai.CameraBoardSocket.CAM_B
)

leftOutput = leftCam.requestOutput(
    size=(CAMERA_WIDTH, CAMERA_HEIGHT),
    type=dai.ImgFrame.Type.GRAY8,
    fps=CAMERA_FPS
)


# ------------------------------------------------------------
# RIGHT MONO CAMERA
# ------------------------------------------------------------

rightCam = pipeline.create(dai.node.Camera).build(
    dai.CameraBoardSocket.CAM_C
)

rightOutput = rightCam.requestOutput(
    size=(CAMERA_WIDTH, CAMERA_HEIGHT),
    type=dai.ImgFrame.Type.GRAY8,
    fps=CAMERA_FPS
)


# ============================================================
# OUTPUT QUEUES
# ============================================================

rgbQueue = rgbOutput.createOutputQueue(
    maxSize=4,
    blocking=False
)

leftQueue = leftOutput.createOutputQueue(
    maxSize=4,
    blocking=False
)

rightQueue = rightOutput.createOutputQueue(
    maxSize=4,
    blocking=False
)


# ============================================================
# START CAMERA
# ============================================================

pipeline.start()

print("OAK-D 3 Camera Test Started")
print("RGB   : CAM_A")
print("LEFT  : CAM_B")
print("RIGHT : CAM_C")
print("Close the camera window [X] to stop")


# ============================================================
# CREATE WINDOW
# ============================================================

cv2.namedWindow(
    WINDOW_NAME,
    cv2.WINDOW_NORMAL
)

cv2.resizeWindow(
    WINDOW_NAME,
    WINDOW_WIDTH,
    WINDOW_HEIGHT
)


# ============================================================
# FPS VARIABLES
# ============================================================

frameCount = 0
startTime = time.time()
fps = 0.0


# ============================================================
# MAIN LOOP
# ============================================================

try:

    while pipeline.isRunning():

        # ----------------------------------------------------
        # Get frames
        # ----------------------------------------------------

        rgbPacket = rgbQueue.get()
        leftPacket = leftQueue.get()
        rightPacket = rightQueue.get()

        rgbFrame = rgbPacket.getCvFrame()
        leftFrame = leftPacket.getCvFrame()
        rightFrame = rightPacket.getCvFrame()


        # ----------------------------------------------------
        # Mono -> BGR
        #
        # เพื่อให้สามารถนำภาพขาวดำมาต่อกับ RGB
        # ด้วย np.hstack() ได้
        # ----------------------------------------------------

        leftFrame = cv2.cvtColor(
            leftFrame,
            cv2.COLOR_GRAY2BGR
        )

        rightFrame = cv2.cvtColor(
            rightFrame,
            cv2.COLOR_GRAY2BGR
        )


        # ----------------------------------------------------
        # Calculate FPS
        # ----------------------------------------------------

        frameCount += 1

        elapsed = time.time() - startTime

        if elapsed >= 1.0:

            fps = frameCount / elapsed

            print(f"FPS: {fps:.2f}")

            frameCount = 0
            startTime = time.time()


        # ----------------------------------------------------
        # Labels
        # ----------------------------------------------------

        cv2.putText(
            leftFrame,
            "LEFT - CAM_B",
            (20, 40),
            cv2.FONT_HERSHEY_SIMPLEX,
            1,
            (255, 255, 255),
            2
        )

        cv2.putText(
            rgbFrame,
            "CENTER RGB - CAM_A",
            (20, 40),
            cv2.FONT_HERSHEY_SIMPLEX,
            1,
            (255, 255, 255),
            2
        )

        cv2.putText(
            rightFrame,
            "RIGHT - CAM_C",
            (20, 40),
            cv2.FONT_HERSHEY_SIMPLEX,
            1,
            (255, 255, 255),
            2
        )


        # ----------------------------------------------------
        # FPS on RGB frame
        # ----------------------------------------------------

        cv2.putText(
            rgbFrame,
            f"FPS: {fps:.1f}",
            (20, 80),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.8,
            (255, 255, 255),
            2
        )


        # ----------------------------------------------------
        # Resize each camera
        #
        # 3 x 420 pixels ~= 1260 pixels
        # ----------------------------------------------------

        displayWidth = 420
        displayHeight = 315

        leftDisplay = cv2.resize(
            leftFrame,
            (displayWidth, displayHeight)
        )

        rgbDisplay = cv2.resize(
            rgbFrame,
            (displayWidth, displayHeight)
        )

        rightDisplay = cv2.resize(
            rightFrame,
            (displayWidth, displayHeight)
        )


        # ----------------------------------------------------
        # Combine 3 cameras horizontally
        # ----------------------------------------------------

        combined = np.hstack(
            (
                leftDisplay,
                rgbDisplay,
                rightDisplay
            )
        )


        # ----------------------------------------------------
        # Create 1280 x 720 canvas
        # ----------------------------------------------------

        canvas = np.zeros(
            (WINDOW_HEIGHT, WINDOW_WIDTH, 3),
            dtype=np.uint8
        )


        # ----------------------------------------------------
        # Center combined image
        # ----------------------------------------------------

        x = (WINDOW_WIDTH - combined.shape[1]) // 2
        y = (WINDOW_HEIGHT - combined.shape[0]) // 2

        canvas[
            y:y + combined.shape[0],
            x:x + combined.shape[1]
        ] = combined


        # ----------------------------------------------------
        # Show
        # ----------------------------------------------------

        cv2.imshow(
            WINDOW_NAME,
            canvas
        )


        # จำเป็นสำหรับ Windows GUI event
        cv2.waitKey(1)


        # ----------------------------------------------------
        # Detect X button
        # ----------------------------------------------------

        try:

            visible = cv2.getWindowProperty(
                WINDOW_NAME,
                cv2.WND_PROP_VISIBLE
            )

            if visible < 1:
                print("Window closed")
                break

        except cv2.error:
            break


# ============================================================
# CLEANUP
# ============================================================

finally:

    print("Stopping OAK-D...")

    pipeline.stop()

    cv2.destroyAllWindows()

    print("OAK-D stopped")

# Test all 3 camera and add depth function

In [2]:
import depthai as dai
import cv2
import numpy as np
import time


# ============================================================
# SETTINGS
# ============================================================

WINDOW_NAME = "OAK-D Full Camera Test"

WINDOW_WIDTH = 1280
WINDOW_HEIGHT = 720

FPS = 30

# Stereo cameras ของ OAK-D
STEREO_SIZE = (640, 400)

# RGB preview
RGB_SIZE = (640, 400)

# ระยะสูงสุดที่ใช้ในการแสดง Depth สี
MAX_DEPTH_MM = 5000


# ============================================================
# PIPELINE
# ============================================================

pipeline = dai.Pipeline()


# ============================================================
# RGB CAMERA - CAM_A
# ============================================================

rgbCam = pipeline.create(dai.node.Camera).build(
    dai.CameraBoardSocket.CAM_A
)

rgbOutput = rgbCam.requestOutput(
    size=RGB_SIZE,
    type=dai.ImgFrame.Type.BGR888p,
    fps=FPS
)


# ============================================================
# LEFT MONO - CAM_B
# ============================================================

leftCam = pipeline.create(dai.node.Camera).build(
    dai.CameraBoardSocket.CAM_B
)

leftOutput = leftCam.requestOutput(
    size=STEREO_SIZE,
    type=dai.ImgFrame.Type.GRAY8,
    fps=FPS
)


# ============================================================
# RIGHT MONO - CAM_C
# ============================================================

rightCam = pipeline.create(dai.node.Camera).build(
    dai.CameraBoardSocket.CAM_C
)

rightOutput = rightCam.requestOutput(
    size=STEREO_SIZE,
    type=dai.ImgFrame.Type.GRAY8,
    fps=FPS
)


# ============================================================
# STEREO DEPTH
# ============================================================

stereo = pipeline.create(dai.node.StereoDepth)

# ช่วยลด depth ผิดพลาด
stereo.setLeftRightCheck(True)

# เพิ่มความละเอียด disparity
stereo.setSubpixel(True)

# ส่ง Left + Right เข้า StereoDepth
leftOutput.link(stereo.left)
rightOutput.link(stereo.right)


# ============================================================
# OUTPUT QUEUES
# ============================================================

rgbQueue = rgbOutput.createOutputQueue(
    maxSize=4,
    blocking=False
)

leftQueue = stereo.rectifiedLeft.createOutputQueue(
    maxSize=4,
    blocking=False
)

rightQueue = stereo.rectifiedRight.createOutputQueue(
    maxSize=4,
    blocking=False
)

depthQueue = stereo.depth.createOutputQueue(
    maxSize=4,
    blocking=False
)


# ============================================================
# START PIPELINE
# ============================================================

pipeline.start()

print("====================================")
print("OAK-D Full Camera Test")
print("====================================")
print("RGB        : CAM_A")
print("Left Mono  : CAM_B")
print("Right Mono : CAM_C")
print("Depth      : Stereo CAM_B + CAM_C")
print("Close window [X] to stop")
print("====================================")


# ============================================================
# WINDOW
# ============================================================

cv2.namedWindow(
    WINDOW_NAME,
    cv2.WINDOW_NORMAL
)

cv2.resizeWindow(
    WINDOW_NAME,
    WINDOW_WIDTH,
    WINDOW_HEIGHT
)

# ============================================================
# FPS
# ============================================================

frameCount = 0
startTime = time.time()
fps = 0.0

# ============================================================
# MAIN LOOP
# ============================================================

try:

    while pipeline.isRunning():

        # ----------------------------------------------------
        # GET DATA
        # ----------------------------------------------------

        rgbPacket = rgbQueue.get()
        leftPacket = leftQueue.get()
        rightPacket = rightQueue.get()
        depthPacket = depthQueue.get()

        rgbFrame = rgbPacket.getCvFrame()

        leftFrame = leftPacket.getCvFrame()
        rightFrame = rightPacket.getCvFrame()

        depthFrame = depthPacket.getFrame()


        # ====================================================
        # FPS
        # ====================================================

        frameCount += 1

        elapsed = time.time() - startTime

        if elapsed >= 1.0:

            fps = frameCount / elapsed

            print(f"FPS: {fps:.2f}")

            frameCount = 0
            startTime = time.time()


        # ====================================================
        # LEFT / RIGHT -> BGR
        # ====================================================

        leftFrame = cv2.cvtColor(
            leftFrame,
            cv2.COLOR_GRAY2BGR
        )

        rightFrame = cv2.cvtColor(
            rightFrame,
            cv2.COLOR_GRAY2BGR
        )


        # ====================================================
        # DEPTH -> COLOR MAP
        # ====================================================

        depthNormalized = np.clip(
            depthFrame,
            0,
            MAX_DEPTH_MM
        )

        depthNormalized = (
            depthNormalized / MAX_DEPTH_MM * 255
        ).astype(np.uint8)

        # กลับค่าเพื่อให้ของใกล้เห็นเด่นขึ้น
        depthNormalized = 255 - depthNormalized

        depthColor = cv2.applyColorMap(
            depthNormalized,
            cv2.COLORMAP_JET
        )

        # Pixel ที่ไม่มี Depth ให้เป็นดำ
        depthColor[depthFrame == 0] = 0


        # ====================================================
        # LABELS
        # ====================================================

        font = cv2.FONT_HERSHEY_SIMPLEX

        cv2.putText(
            leftFrame,
            "LEFT MONO - CAM_B",
            (20, 35),
            font,
            0.8,
            (255, 255, 255),
            2
        )

        cv2.putText(
            rgbFrame,
            "RGB - CAM_A",
            (20, 35),
            font,
            0.8,
            (255, 255, 255),
            2
        )

        cv2.putText(
            rightFrame,
            "RIGHT MONO - CAM_C",
            (20, 35),
            font,
            0.8,
            (255, 255, 255),
            2
        )

        cv2.putText(
            depthColor,
            "STEREO DEPTH",
            (20, 35),
            font,
            0.8,
            (255, 255, 255),
            2
        )

        cv2.putText(
            rgbFrame,
            f"FPS: {fps:.1f}",
            (20, 70),
            font,
            0.7,
            (255, 255, 255),
            2
        )


        # ====================================================
        # RESIZE EACH PANEL
        # ====================================================

        PANEL_WIDTH = 640
        PANEL_HEIGHT = 360

        leftDisplay = cv2.resize(
            leftFrame,
            (PANEL_WIDTH, PANEL_HEIGHT)
        )

        rgbDisplay = cv2.resize(
            rgbFrame,
            (PANEL_WIDTH, PANEL_HEIGHT)
        )

        rightDisplay = cv2.resize(
            rightFrame,
            (PANEL_WIDTH, PANEL_HEIGHT)
        )

        depthDisplay = cv2.resize(
            depthColor,
            (PANEL_WIDTH, PANEL_HEIGHT)
        )

        # ====================================================
        # 2 x 2 LAYOUT
        # ====================================================

        topRow = np.hstack(
            (
                leftDisplay,
                rgbDisplay
            )
        )

        bottomRow = np.hstack(
            (
                rightDisplay,
                depthDisplay
            )
        )

        canvas = np.vstack(
            (
                topRow,
                bottomRow
            )
        )

        # ====================================================
        # SHOW
        # ====================================================

        cv2.imshow(
            WINDOW_NAME,
            canvas
        )

        # ให้ Windows ประมวลผล GUI event
        cv2.waitKey(1)

        # ====================================================
        # CLOSE USING X BUTTON
        # ====================================================

        try:

            visible = cv2.getWindowProperty(
                WINDOW_NAME,
                cv2.WND_PROP_VISIBLE
            )

            if visible < 1:
                print("Window closed.")
                break

        except cv2.error:
            break


# ============================================================
# CLEANUP
# ============================================================

finally:

    print("Stopping OAK-D...")

    pipeline.stop()

    cv2.destroyAllWindows()

    print("OAK-D stopped.")

OAK-D Full Camera Test
RGB        : CAM_A
Left Mono  : CAM_B
Right Mono : CAM_C
Depth      : Stereo CAM_B + CAM_C
Close window [X] to stop
FPS: 19.47
FPS: 18.24
FPS: 16.70
FPS: 15.76
FPS: 16.20
FPS: 15.73
FPS: 15.95
FPS: 17.98
FPS: 16.20
FPS: 16.38
FPS: 15.92
FPS: 15.83
FPS: 15.82
FPS: 15.57
FPS: 16.81
FPS: 16.46
FPS: 17.11
FPS: 16.48
FPS: 16.28
FPS: 17.70
FPS: 15.46
FPS: 17.61
FPS: 16.61
FPS: 18.51
FPS: 17.87
FPS: 16.59
FPS: 16.81
FPS: 16.83
FPS: 15.97
FPS: 16.23
FPS: 16.12
FPS: 16.88
FPS: 15.06
FPS: 16.92
FPS: 15.90
FPS: 15.93
FPS: 17.10
FPS: 16.19
FPS: 16.92
FPS: 16.07
FPS: 15.97
FPS: 15.84
FPS: 16.88
FPS: 15.75
FPS: 17.06
FPS: 17.67
FPS: 16.89
FPS: 17.46
FPS: 16.55
FPS: 16.24
FPS: 17.76
FPS: 16.25
FPS: 16.98
FPS: 15.20
FPS: 16.90
FPS: 17.27
FPS: 16.91
FPS: 17.32
FPS: 17.81
FPS: 16.00
FPS: 17.59
FPS: 17.08
FPS: 16.90
FPS: 15.81
FPS: 17.26
FPS: 16.21
FPS: 17.92
FPS: 16.88
FPS: 16.48
FPS: 15.37
FPS: 15.82
FPS: 17.81
FPS: 16.67
FPS: 16.68
FPS: 16.67
FPS: 16.66
FPS: 16.80
FPS: 17.77
FPS